# Fix Neural Data from MATLAB Files

This notebook loads sorted spikes data from MATLAB files and creates a DataFrame with trial names and neural data.

In [1]:
import numpy as np
import pandas as pd
import re
from scipy.io import loadmat
from pathlib import Path
from tqdm import tqdm

monkey_name = 'fiona' # 'fiona' or 'yasmin'

In [2]:
def parse_neural_data_from_mat(data):
    """
    Parse MATLAB dataStruct into a DataFrame with trial names and neural data.
    
    Parameters:
    -----------
    data : dict
        Dictionary loaded from MATLAB file using scipy.io.loadmat
    
    Returns:
    --------
    pd.DataFrame : DataFrame with columns ['trial_name', 'neural_data']
        neural_data is a dict with keys=neuron_id (int), values=spike_times (numpy array)
    """
    data_struct = data['dataStruct']
    num_trials = data_struct.shape[1]
    
    trial_names = []
    neural_data_list = []
    
    for i in range(num_trials):
        trial_entry = data_struct[0, i]
        
        # Extract trial name
        trial_name_obj = trial_entry[0]
        if hasattr(trial_name_obj, '__len__') and len(trial_name_obj) > 0:
            trial_name = str(trial_name_obj[0])
        else:
            raise ValueError(f"Unexpected trial name structure in trial index {i}")
            # trial_name = f"trial_{i}"
        
        # Extract spikes for all neurons
        spikes_obj = trial_entry[1]
        neural_data_dict = {}
        
        if hasattr(spikes_obj, 'shape') and len(spikes_obj.shape) > 1:
            for idx, neuron_spikes in enumerate(spikes_obj[0]):
                # Flatten the spike times array and add them to the dict
                neural_data_dict[idx] = neuron_spikes.flatten()
            # num_neurons = spikes_obj.shape[1]
            
            # for j in range(num_neurons):
            #     neuron_spikes = spikes_obj[0, j]
                
            #     # Flatten the spike times array
            #     if hasattr(neuron_spikes, 'flatten'):
            #         spike_times = neuron_spikes.flatten()
            #         # Only add non-empty spike trains
            #         if len(spike_times) > 0:
            #             neural_data_dict[j] = spike_times
            #     elif hasattr(neuron_spikes, '__len__') and len(neuron_spikes) > 0:
            #         spike_times = np.array(neuron_spikes).flatten()
            #         if len(spike_times) > 0:
            #             neural_data_dict[j] = spike_times
            #     else:
            #         raise ValueError(f"Unexpected spike times structure for neuron {j} in trial {trial_name}")
        else:
            raise ValueError(f"Unexpected spikes object structure in trial {trial_name}")
        trial_names.append(trial_name)
        neural_data_list.append(neural_data_dict)

    
    # Create DataFrame
    df = pd.DataFrame({
        'trial_name': trial_names,
        'neural_data': neural_data_list
    })
    
    return df


In [3]:
# Load the exemplar file
data_dir = Path.cwd().parent / 'data' / 'fiona_sst' / 'sorted_spikes_in_session'
mat_file = data_dir / 'fi211109_sorted_spikes.mat'

print(f"Loading: {mat_file}")
print(f"File exists: {mat_file.exists()}")

if mat_file.exists():
    data = loadmat(mat_file)
    print(f"\nTop-level keys: {list(data.keys())}")
    print(f"\ndataStruct shape: {data['dataStruct'].shape}")
    print(f"dataStruct dtype: {data['dataStruct'].dtype}")
    print(f"\nNumber of trials: {data['dataStruct'].shape[1]}")


# Parse examplar data
neural_df = parse_neural_data_from_mat(data)

print(f"Created DataFrame with shape: {neural_df.shape}")
print(f"\nFirst few rows:")
print(neural_df.head())
print(f"\nSample neural_data for first trial:")
first_trial_data = neural_df.iloc[0]['neural_data']
print(f"  Number of neurons with spikes: {len(first_trial_data)}")
print(f"  Neuron IDs: {list(first_trial_data.keys())[:10]}...")  # Show first 10
if len(first_trial_data) > 0:
    first_neuron_id = list(first_trial_data.keys())[0]
    print(f"  Example - Neuron {first_neuron_id} spike times: {first_trial_data[first_neuron_id]}")

Loading: /home/barak/Projects/population-analysis/data/fiona_sst/sorted_spikes_in_session/fi211109_sorted_spikes.mat
File exists: True

Top-level keys: ['__header__', '__version__', '__globals__', 'dataStruct']

dataStruct shape: (1, 2040)
dataStruct dtype: [('trial', 'O'), ('spikes', 'O')]

Number of trials: 2040
Created DataFrame with shape: (2040, 2)

First few rows:
       trial_name                                        neural_data
0  fi211109a.0001  {0: [], 1: [1510.14, 1632.64], 2: [], 3: [], 4...
1  fi211109a.0002  {0: [1196.53], 1: [455.41, 1398.74, 1966.24, 2...
2  fi211109a.0003  {0: [1229.15, 1318.25], 1: [961.06], 2: [], 3:...
3  fi211109a.0004  {0: [2339.12], 1: [326.66, 972.53], 2: [], 3: ...
4  fi211109a.0005  {0: [1033.38, 1055.85], 1: [875.09, 2149.34], ...

Sample neural_data for first trial:
  Number of neurons with spikes: 200
  Neuron IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]...
  Example - Neuron 0 spike times: []


In [4]:
data_struct = data['dataStruct']
num_trials = data_struct.shape[1]

trial_entry = data_struct[0, 0]
trial_name_obj = trial_entry[0]
spikes_obj = trial_entry[1]
num_neurons = spikes_obj.shape[1]

neuron_spikes = spikes_obj[0, 0]

spike_times = neuron_spikes.flatten()

spikes_obj[0][0]

array([], shape=(1, 0), dtype=float64)

In [5]:
# Aggregate neural data from all '*_sorted_spikes.mat' files under data/

data_root = Path.cwd().parent / 'data' / f'{monkey_name}_sst' / 'sorted_spikes_in_session'
pattern = f'{monkey_name[:2]}2*_sorted_spikes.mat'
all_files = sorted(list(data_root.rglob(pattern)))
print(f'Found {len(all_files)} files matching pattern "{pattern}" under {data_root}')

# Filter out hidden/system files (starting with ._) and validate session name format
session_pattern = re.compile(r'^(?:fi|ya)2\d{5}$')
valid_files = []
for f in all_files:
    session_name = f.stem.split('_sorted_spikes')[0]
    if ((f.name.startswith('._')) or (not session_pattern.match(session_name))):
        print(f'Skipping hidden/system file: {f.name}')
        continue
    valid_files.append(f)

print(f'\nProcessing {len(valid_files)} valid files of {len(all_files)} files found:\n')

dfs = []
for mat_file in tqdm(valid_files, desc='Processing files'):
    # print(f'Processing: {mat_file.relative_to(data_root)}')
    try:
        data_local = loadmat(mat_file)
    except Exception as e:
        print(f'  Failed to load {mat_file.name}: {e}')
        continue

    # Parse using the function defined earlier in this notebook
    df = parse_neural_data_from_mat(data_local)

    # Derive session name from filename (strip suffix '_sorted_spikes')
    session_name = mat_file.stem.split('_sorted_spikes')[0]
    df['session'] = session_name

    # Ensure trial_number exists as integer on each parsed df
    if 'trial_number' not in df.columns:
        df['trial_number'] = df['trial_name'].apply(lambda x: int(x.split('.')[-1]))
    else:
        # coerce to int if strings with leading zeros are present
        df['trial_number'] = df['trial_number'].astype(int)

    dfs.append(df)

# Concatenate all session DataFrames
if len(dfs) > 0:
    neural_df_all = pd.concat(dfs, ignore_index=True)
    print(f'Combined DataFrame shape: {neural_df_all.shape}')
    print(f'Total sessions: {neural_df_all["session"].unique().shape[0]}')
    print(f'Total trials: {len(neural_df_all)}')
else:
    print('No parsed files found; neural_df_all not created')

Found 88 files matching pattern "fi2*_sorted_spikes.mat" under /home/barak/Projects/population-analysis/data/fiona_sst/sorted_spikes_in_session

Processing 88 valid files of 88 files found:



Processing files:   0%|          | 0/88 [00:00<?, ?it/s]

Processing files: 100%|██████████| 88/88 [00:13<00:00,  6.44it/s]

Combined DataFrame shape: (126165, 4)
Total sessions: 88
Total trials: 126165


In [6]:
len(valid_files)

88

In [7]:
print("Neurons per trial: ",
    neural_df_all.apply(
        lambda row: len(row['neural_data'].keys()), 
        axis=1
    ).value_counts().index[0]
)
neural_df_all.head()

Neurons per trial:  200


,trial_name,neural_data,session,trial_number
0,fi210628a.0001,"{0: [28.44, 60.23, 94.46, 164.23, 193.88, 223....",fi210628,1
1,fi210628a.0002,"{0: [18.74, 59.81, 90.46, 131.34, 170.34, 202....",fi210628,2
2,fi210628a.0003,"{0: [5.49, 88.89, 178.81, 380.84, 480.36, 608....",fi210628,3
3,fi210628a.0004,"{0: [70.66, 437.66, 656.51, 696.26, 754.99, 81...",fi210628,4
4,fi210628a.0005,"{0: [66.44, 127.21, 154.31, 210.86, 235.24, 26...",fi210628,5


In [8]:
neurons_list = []

def extract_neuron_ids(row, neurons_list):
    trial_neuron_ids = [f"{row['session']}_{key}" for key in row['neural_data'].keys()]
    neurons_list += trial_neuron_ids
    return trial_neuron_ids

neural_df_all.apply(
    lambda row: extract_neuron_ids(row, neurons_list), 
    axis=1
)

neurons_set = set(neurons_list)
print(f'Total unique neurons across all sessions: {len(neurons_set)}')

Total unique neurons across all sessions: 17600


In [9]:
1307+1794+579+113

3793

In [11]:
base_path = Path.cwd().parent / 'data' 
# file_path = base_path / 'csst_trials_pkls' / f'all_{monkey_name}_CSST_trials_df.pkl'
file_path = base_path / 'csst_trials_pkls' / f'no_filters_{monkey_name}_CSST_trials_df.pkl'

orig_df = pd.read_pickle(file_path)
orig_df

,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type,vPos,vVel,reaction_time
0,None,0.0,R,fi210713a.0529,NaN,11278,921,"[28.55, 28.55, 28.55, 28.55, 28.55, 28.55, 28....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",None,...,1005.0,False,1605,STOP_R_SSD2,0529,fi210713a,STOP,"[-11.575, -11.575, -11.575, -11.6, -11.6, -11....","[-4.686380092992484, -4.686380092992484, -4.04...",NaN
1,None,0.0,R,fi210713a.0480,NaN,11278,934,"[-0.9, -0.9, -0.9, -0.95, -0.95, -0.9, -0.925,...","[2.4810247551136677, 2.4810247551136677, 0.459...",None,...,958.0,False,1558,STOP_R_SSD1,0480,fi210713a,STOP,"[-0.575, -0.575, -0.5, -0.475, -0.475, -0.5, -...","[-6.6160660136364475, -6.6160660136364475, -2....",NaN
2,"[338, 444, 447, 449]",180.0,L,fi210713a.0457,NaN,11278,1093,"[-11.775, -11.775, -11.775, -11.775, -11.775, ...","[-1.286457280429309, -1.286457280429309, -0.91...",None,...,1117.0,False,1717,STOP_L_SSD1,0457,fi210713a,STOP,"[-0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0....","[-3.6755922297980264, -3.6755922297980264, -1....",NaN
3,None,0.0,R,fi210713a.0169,"[1134, 1214]",9222,921,"[-13.225, -13.225, -13.25, -13.175, -13.175, -...","[-3.951261647032878, -3.951261647032878, -5.51...",None,...,1125.0,True,1726,STOP_R_SSD4,0169,fi210713a,STOP,"[-0.5, -0.5, -0.525, -0.525, -0.525, -0.5, -0....","[-1.3783470861742597, -1.3783470861742597, -0....",213.0
4,None,180.0,L,fi210713a.0688,"[1150, 1224]",8206,967,"[12.45, 12.45, 12.425, 12.425, 12.4, 12.4, 12....","[-0.27566941723485194, -0.27566941723485194, -...",None,...,NaN,False,2118,GO_L,0688,fi210713a,GO,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.18377961148990132, 0.18377961148990132, -1....",183.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126160,None,NaN,None,fi211110a.1099,"[1563, 1633]",8334,1370,"[-0.2, -0.2, -0.2, -0.2, -0.2, -0.2, -0.25, -0...","[1.8377961148990132, 1.8377961148990132, 2.113...","{0: [54.03, 136.98000000000002, 478.0900000000...",...,NaN,False,2650,d135,1099,fi211110a,None,"[-6.6, -6.6, -6.65, -6.65, -6.65, -6.7, -6.7, ...","[5.053939315972286, 5.053939315972286, 3.85937...",193.0
126161,"[0, 34]",0.0,R,fi211110a.1953,"[1141, 1215]",8206,972,"[6.425, 6.425, 6.425, 6.425, 6.425, 6.425, 6.4...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","{0: [275.24, 540.29, 597.81, 739.29, 919.16, 9...",...,1200.0,False,2123,CONT_R_SSD4,1953,fi211110a,CONT,"[-27.55, -27.55, -27.55, -27.55, -27.55, -27.5...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",169.0
126162,None,180.0,L,fi211110a.0735,"[1291, 1363]",8206,1003,"[11.2, 11.2, 11.2, 11.175, 11.175, 11.3, 11.27...","[-0.6432286402146545, -0.6432286402146545, -1....","{0: [251.54, 291.23, 298.21000000000004, 472.3...",...,1051.0,False,2154,CONT_L_SSD1,0735,fi211110a,CONT,"[-0.3, -0.3, -0.3, -0.275, -0.275, -0.275, -0....","[-1.1945674746843584, -1.1945674746843584, 1.1...",288.0
126163,None,0.0,R,fi211110a.1550,"[1782, 1823]",11278,1016,"[11.425, 11.425, 11.425, 11.45, 11.45, 11.4, 1...","[-2.572914560858618, -2.572914560858618, -2.29...","{0: [426.89, 918.69, 1329.6100000000001, 1748....",...,1184.0,False,1884,STOP_R_SSD3,1550,fi211110a,STOP,"[-0.725, -0.725, -0.725, -0.825, -0.825, -0.82...","[4.502600481502582, 4.502600481502582, 3.85937...",766.0


In [12]:
cell_ids = set([])
cell_ids_list = orig_df['neural_data'].apply(
    lambda x: list(x.keys()) if isinstance(x, dict) else x
)

for row in cell_ids_list:
    if isinstance(row, list):
        cell_ids.update(row)

lst = [i for i in cell_ids]
lst.sort()
lst == list(range(0, 50))

True

In [13]:
# Join neural_df_all with orig_df
# Select only trial_name and neural_data from neural_df_all
neural_subset = neural_df_all[['trial_name', 'neural_data']].copy()

# Rename neural_data to new_neural_data before merging
neural_subset.rename(columns={'neural_data': 'new_neural_data'}, inplace=True)

# Merge on trial_name (neural_df_all) = filename (orig_df)
merged_df = orig_df.merge(
    neural_subset,
    left_on='filename',
    right_on='trial_name',
    how='left'
)

print(f"Original df shape: {orig_df.shape}")
print(f"Neural df shape: {neural_df_all.shape}")
print(f"Merged df shape: {merged_df.shape}")
print(f"\nMerged df columns: {list(merged_df.columns)}")
print(f"\nRows with new_neural_data: {merged_df['new_neural_data'].notna().sum()}")
print(f"Rows without new_neural_data: {merged_df['new_neural_data'].isna().sum()}")

merged_df

Original df shape: (126165, 28)
Neural df shape: (126165, 4)
Merged df shape: (126165, 30)

Merged df columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'neural_data', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name_x', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel', 'reaction_time', 'trial_name_y', 'new_neural_data']

Rows with new_neural_data: 126165
Rows without new_neural_data: 0


,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,trial_length,trial_name_x,trial_number,trial_session,type,vPos,vVel,reaction_time,trial_name_y,new_neural_data
0,None,0.0,R,fi210713a.0529,NaN,11278,921,"[28.55, 28.55, 28.55, 28.55, 28.55, 28.55, 28....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",None,...,1605,STOP_R_SSD2,0529,fi210713a,STOP,"[-11.575, -11.575, -11.575, -11.6, -11.6, -11....","[-4.686380092992484, -4.686380092992484, -4.04...",NaN,fi210713a.0529,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ..."
1,None,0.0,R,fi210713a.0480,NaN,11278,934,"[-0.9, -0.9, -0.9, -0.95, -0.95, -0.9, -0.925,...","[2.4810247551136677, 2.4810247551136677, 0.459...",None,...,1558,STOP_R_SSD1,0480,fi210713a,STOP,"[-0.575, -0.575, -0.5, -0.475, -0.475, -0.5, -...","[-6.6160660136364475, -6.6160660136364475, -2....",NaN,fi210713a.0480,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ..."
2,"[338, 444, 447, 449]",180.0,L,fi210713a.0457,NaN,11278,1093,"[-11.775, -11.775, -11.775, -11.775, -11.775, ...","[-1.286457280429309, -1.286457280429309, -0.91...",None,...,1717,STOP_L_SSD1,0457,fi210713a,STOP,"[-0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0....","[-3.6755922297980264, -3.6755922297980264, -1....",NaN,fi210713a.0457,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ..."
3,None,0.0,R,fi210713a.0169,"[1134, 1214]",9222,921,"[-13.225, -13.225, -13.25, -13.175, -13.175, -...","[-3.951261647032878, -3.951261647032878, -5.51...",None,...,1726,STOP_R_SSD4,0169,fi210713a,STOP,"[-0.5, -0.5, -0.525, -0.525, -0.525, -0.5, -0....","[-1.3783470861742597, -1.3783470861742597, -0....",213.0,fi210713a.0169,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ..."
4,None,180.0,L,fi210713a.0688,"[1150, 1224]",8206,967,"[12.45, 12.45, 12.425, 12.425, 12.4, 12.4, 12....","[-0.27566941723485194, -0.27566941723485194, -...",None,...,2118,GO_L,0688,fi210713a,GO,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.18377961148990132, 0.18377961148990132, -1....",183.0,fi210713a.0688,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126160,None,NaN,None,fi211110a.1099,"[1563, 1633]",8334,1370,"[-0.2, -0.2, -0.2, -0.2, -0.2, -0.2, -0.25, -0...","[1.8377961148990132, 1.8377961148990132, 2.113...","{0: [54.03, 136.98000000000002, 478.0900000000...",...,2650,d135,1099,fi211110a,None,"[-6.6, -6.6, -6.65, -6.65, -6.65, -6.7, -6.7, ...","[5.053939315972286, 5.053939315972286, 3.85937...",193.0,fi211110a.1099,"{0: [54.03, 136.98, 478.09, 710.21, 895.96, 10..."
126161,"[0, 34]",0.0,R,fi211110a.1953,"[1141, 1215]",8206,972,"[6.425, 6.425, 6.425, 6.425, 6.425, 6.425, 6.4...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","{0: [275.24, 540.29, 597.81, 739.29, 919.16, 9...",...,2123,CONT_R_SSD4,1953,fi211110a,CONT,"[-27.55, -27.55, -27.55, -27.55, -27.55, -27.5...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",169.0,fi211110a.1953,"{0: [275.24, 540.29, 597.81, 739.29, 919.16, 9..."
126162,None,180.0,L,fi211110a.0735,"[1291, 1363]",8206,1003,"[11.2, 11.2, 11.2, 11.175, 11.175, 11.3, 11.27...","[-0.6432286402146545, -0.6432286402146545, -1....","{0: [251.54, 291.23, 298.21000000000004, 472.3...",...,2154,CONT_L_SSD1,0735,fi211110a,CONT,"[-0.3, -0.3, -0.3, -0.275, -0.275, -0.275, -0....","[-1.1945674746843584, -1.1945674746843584, 1.1...",288.0,fi211110a.0735,"{0: [251.54, 291.23, 298.21, 472.31, 521.23, 6..."
126163,None,0.0,R,fi211110a.1550,"[1782, 1823]",11278,1016,"[11.425, 11.425, 11.425, 11.45, 11.45, 11.4, 1...","[-2.572914560858618, -2.572914560858618, -2.29...","{0: [426.89, 918.69, 1329.6100000000001, 1748....",...,1884,STOP_R_SSD3,1550,fi211110a,STOP,"[-0.725, -0.725, -0.725, -0.825, -0.825, -0.82...","[4.502600481502582, 4.502600481502582, 3.85937...",766.0,fi211110a.1550,"{0: [426.89, 918.69, 1329.61, 1748.56, 1755.09..."


In [14]:
# Drop old neural_data and trial_name_y columns, rename new_neural_data to neural_data
merged_df = merged_df.drop(columns=['neural_data', 'trial_name_y'])
merged_df = merged_df.rename(columns={'new_neural_data': 'neural_data'})

print(f"Updated merged_df shape: {merged_df.shape}")
print(f"Updated columns: {list(merged_df.columns)}")

merged_df

Updated merged_df shape: (126165, 28)
Updated columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name_x', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel', 'reaction_time', 'neural_data']


,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,saccades,...,trial_failed,trial_length,trial_name_x,trial_number,trial_session,type,vPos,vVel,reaction_time,neural_data
0,None,0.0,R,fi210713a.0529,NaN,11278,921,"[28.55, 28.55, 28.55, 28.55, 28.55, 28.55, 28....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[227, 318], [545, 603]]",...,False,1605,STOP_R_SSD2,0529,fi210713a,STOP,"[-11.575, -11.575, -11.575, -11.6, -11.6, -11....","[-4.686380092992484, -4.686380092992484, -4.04...",NaN,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ..."
1,None,0.0,R,fi210713a.0480,NaN,11278,934,"[-0.9, -0.9, -0.9, -0.95, -0.95, -0.9, -0.925,...","[2.4810247551136677, 2.4810247551136677, 0.459...","[[386, 442]]",...,False,1558,STOP_R_SSD1,0480,fi210713a,STOP,"[-0.575, -0.575, -0.5, -0.475, -0.475, -0.5, -...","[-6.6160660136364475, -6.6160660136364475, -2....",NaN,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ..."
2,"[338, 444, 447, 449]",180.0,L,fi210713a.0457,NaN,11278,1093,"[-11.775, -11.775, -11.775, -11.775, -11.775, ...","[-1.286457280429309, -1.286457280429309, -0.91...","[[110, 184], [299, 370], [424, 544], [513, 569]]",...,False,1717,STOP_L_SSD1,0457,fi210713a,STOP,"[-0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0....","[-3.6755922297980264, -3.6755922297980264, -1....",NaN,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ..."
3,None,0.0,R,fi210713a.0169,"[1134, 1214]",9222,921,"[-13.225, -13.225, -13.25, -13.175, -13.175, -...","[-3.951261647032878, -3.951261647032878, -5.51...","[[16, 89], [344, 416], [881, 936], [1134, 1214]]",...,True,1726,STOP_R_SSD4,0169,fi210713a,STOP,"[-0.5, -0.5, -0.525, -0.525, -0.525, -0.5, -0....","[-1.3783470861742597, -1.3783470861742597, -0....",213.0,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ..."
4,None,180.0,L,fi210713a.0688,"[1150, 1224]",8206,967,"[12.45, 12.45, 12.425, 12.425, 12.4, 12.4, 12....","[-0.27566941723485194, -0.27566941723485194, -...","[[257, 335], [1150, 1224], [1909, 1965]]",...,False,2118,GO_L,0688,fi210713a,GO,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.18377961148990132, 0.18377961148990132, -1....",183.0,"{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126160,None,NaN,None,fi211110a.1099,"[1563, 1633]",8334,1370,"[-0.2, -0.2, -0.2, -0.2, -0.2, -0.2, -0.25, -0...","[1.8377961148990132, 1.8377961148990132, 2.113...","[[201, 269], [798, 848], [1298, 1348], [1563, ...",...,False,2650,d135,1099,fi211110a,None,"[-6.6, -6.6, -6.65, -6.65, -6.65, -6.7, -6.7, ...","[5.053939315972286, 5.053939315972286, 3.85937...",193.0,"{0: [54.03, 136.98, 478.09, 710.21, 895.96, 10..."
126161,"[0, 34]",0.0,R,fi211110a.1953,"[1141, 1215]",8206,972,"[6.425, 6.425, 6.425, 6.425, 6.425, 6.425, 6.4...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[14, 154], [127, 193], [501, 558], [1141, 121...",...,False,2123,CONT_R_SSD4,1953,fi211110a,CONT,"[-27.55, -27.55, -27.55, -27.55, -27.55, -27.5...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",169.0,"{0: [275.24, 540.29, 597.81, 739.29, 919.16, 9..."
126162,None,180.0,L,fi211110a.0735,"[1291, 1363]",8206,1003,"[11.2, 11.2, 11.2, 11.175, 11.175, 11.3, 11.27...","[-0.6432286402146545, -0.6432286402146545, -1....","[[147, 224], [734, 793], [1291, 1363], [1771, ...",...,False,2154,CONT_L_SSD1,0735,fi211110a,CONT,"[-0.3, -0.3, -0.3, -0.275, -0.275, -0.275, -0....","[-1.1945674746843584, -1.1945674746843584, 1.1...",288.0,"{0: [251.54, 291.23, 298.21, 472.31, 521.23, 6..."
126163,None,0.0,R,fi211110a.1550,"[1782, 1823]",11278,1016,"[11.425, 11.425, 11.425, 11.45, 11.45, 11.4, 1...","[-2.572914560858618, -2.572914560858618, -2.29...","[[100, 175], [1782, 1823]]",...,False,1884,STOP_R_SSD3,1550,fi211110a,STOP,"[-0.725, -0.725, -0.725, -0.825, -0.825, -0.82...","[4.502600481502582, 4.502600481502582, 3.85937...",766.0,"{0: [426.89, 918.69, 1329.61, 1748.56, 1755.09..."


In [15]:
# Rename trial_name_x to trial_name
merged_df = merged_df.rename(columns={'trial_name_x': 'trial_name'})

# Reorder columns to match orig_df
orig_columns = list(orig_df.columns)
merged_df = merged_df[orig_columns]

print(f"Updated merged_df shape: {merged_df.shape}")
print(f"Original df columns: {list(orig_df.columns)}")
print(f"Merged df columns: {list(merged_df.columns)}")
print(f"\nColumns match: {list(orig_df.columns) == list(merged_df.columns)}")

merged_df

Updated merged_df shape: (126165, 28)
Original df columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'neural_data', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel', 'reaction_time']
Merged df columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'neural_data', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel', 'reaction_time']

Columns match: True


,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type,vPos,vVel,reaction_time
0,None,0.0,R,fi210713a.0529,NaN,11278,921,"[28.55, 28.55, 28.55, 28.55, 28.55, 28.55, 28....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...",...,1005.0,False,1605,STOP_R_SSD2,0529,fi210713a,STOP,"[-11.575, -11.575, -11.575, -11.6, -11.6, -11....","[-4.686380092992484, -4.686380092992484, -4.04...",NaN
1,None,0.0,R,fi210713a.0480,NaN,11278,934,"[-0.9, -0.9, -0.9, -0.95, -0.95, -0.9, -0.925,...","[2.4810247551136677, 2.4810247551136677, 0.459...","{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...",...,958.0,False,1558,STOP_R_SSD1,0480,fi210713a,STOP,"[-0.575, -0.575, -0.5, -0.475, -0.475, -0.5, -...","[-6.6160660136364475, -6.6160660136364475, -2....",NaN
2,"[338, 444, 447, 449]",180.0,L,fi210713a.0457,NaN,11278,1093,"[-11.775, -11.775, -11.775, -11.775, -11.775, ...","[-1.286457280429309, -1.286457280429309, -0.91...","{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...",...,1117.0,False,1717,STOP_L_SSD1,0457,fi210713a,STOP,"[-0.45, -0.45, -0.45, -0.45, -0.45, -0.45, -0....","[-3.6755922297980264, -3.6755922297980264, -1....",NaN
3,None,0.0,R,fi210713a.0169,"[1134, 1214]",9222,921,"[-13.225, -13.225, -13.25, -13.175, -13.175, -...","[-3.951261647032878, -3.951261647032878, -5.51...","{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...",...,1125.0,True,1726,STOP_R_SSD4,0169,fi210713a,STOP,"[-0.5, -0.5, -0.525, -0.525, -0.525, -0.5, -0....","[-1.3783470861742597, -1.3783470861742597, -0....",213.0
4,None,180.0,L,fi210713a.0688,"[1150, 1224]",8206,967,"[12.45, 12.45, 12.425, 12.425, 12.4, 12.4, 12....","[-0.27566941723485194, -0.27566941723485194, -...","{0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: ...",...,NaN,False,2118,GO_L,0688,fi210713a,GO,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.18377961148990132, 0.18377961148990132, -1....",183.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126160,None,NaN,None,fi211110a.1099,"[1563, 1633]",8334,1370,"[-0.2, -0.2, -0.2, -0.2, -0.2, -0.2, -0.25, -0...","[1.8377961148990132, 1.8377961148990132, 2.113...","{0: [54.03, 136.98, 478.09, 710.21, 895.96, 10...",...,NaN,False,2650,d135,1099,fi211110a,None,"[-6.6, -6.6, -6.65, -6.65, -6.65, -6.7, -6.7, ...","[5.053939315972286, 5.053939315972286, 3.85937...",193.0
126161,"[0, 34]",0.0,R,fi211110a.1953,"[1141, 1215]",8206,972,"[6.425, 6.425, 6.425, 6.425, 6.425, 6.425, 6.4...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","{0: [275.24, 540.29, 597.81, 739.29, 919.16, 9...",...,1200.0,False,2123,CONT_R_SSD4,1953,fi211110a,CONT,"[-27.55, -27.55, -27.55, -27.55, -27.55, -27.5...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",169.0
126162,None,180.0,L,fi211110a.0735,"[1291, 1363]",8206,1003,"[11.2, 11.2, 11.2, 11.175, 11.175, 11.3, 11.27...","[-0.6432286402146545, -0.6432286402146545, -1....","{0: [251.54, 291.23, 298.21, 472.31, 521.23, 6...",...,1051.0,False,2154,CONT_L_SSD1,0735,fi211110a,CONT,"[-0.3, -0.3, -0.3, -0.275, -0.275, -0.275, -0....","[-1.1945674746843584, -1.1945674746843584, 1.1...",288.0
126163,None,0.0,R,fi211110a.1550,"[1782, 1823]",11278,1016,"[11.425, 11.425, 11.425, 11.45, 11.45, 11.4, 1...","[-2.572914560858618, -2.572914560858618, -2.29...","{0: [426.89, 918.69, 1329.61, 1748.56, 1755.09...",...,1184.0,False,1884,STOP_R_SSD3,1550,fi211110a,STOP,"[-0.725, -0.725, -0.725, -0.825, -0.825, -0.82...","[4.502600481502582, 4.502600481502582, 3.85937...",766.0


In [16]:
neurons_list = []

def extract_neuron_ids(row, neurons_list):
    trial_neuron_ids = [f"{row['trial_session']}_{key}" for key in row['neural_data'].keys()]
    neurons_list += trial_neuron_ids
    return trial_neuron_ids

merged_df.apply(
    lambda row: extract_neuron_ids(row, neurons_list), 
    axis=1
)

neurons_set = set(neurons_list)
print(f'Total unique neurons across all sessions: {len(neurons_set)}')

Total unique neurons across all sessions: 17600


In [20]:
# file_path = base_path / 'csst_trials_pkls' / f'all_{monkey_name}_CSST_trials_df_including_all_200_neuron_fields.pkl'
file_path = base_path / 'csst_trials_pkls' / f'no_filters_{monkey_name}_CSST_trials_df_including_all_200_neuron_fields.pkl'
print(f"Saving updated DataFrame to: {file_path}")

Saving updated DataFrame to: /home/barak/Projects/population-analysis/data/csst_trials_pkls/no_filters_fiona_CSST_trials_df_including_all_200_neuron_fields.pkl


In [21]:
merged_df.to_pickle(file_path)

In [19]:
# monkey = 'fiona'
# save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
# pickle_file = save_path / f'unified_{monkey}_cell_trial_data.pkl'

# cells_db = pd.read_pickle(pickle_file)
# cells_db['maestro_ID'].value_counts().sort_index()